# PKG staging — cross-source duplicate transactions in `neo4j_payments`

**Question.** Is one real payment landing twice under two categories (e.g. `5C.RTP_PRT_CPTY_PAYS` and
`6C.RTP_P2P_CPTY_PAYS`) with different `trans_id`s?

**Duplicate key.** same PNC-side account + same role (payer/receiver) + same amount (exact cents) + same `trans_dt`.
Category and `trans_id` are deliberately *not* in the key, since those are what differ between the two sources.

**How the result is judged.** A same-day match rate means nothing on its own, because a customer can genuinely receive two
$50 payments on one day. Every match rate is therefore shown next to a **placebo**: the same test with dates shifted
±7 days. The duplicate signal is the same-day rate *minus* the placebo rate.

| § | What | Output |
|---|---|---|
| 0 | Schema, partition layout (verifies the category + date partitioning), window | metadata only |
| 1 | Window QA: row count, `trans_id` uniqueness, anchors | 2 tables |
| 2 | Category inventory: direction derived from nulls, src_syst, rail | 2 tables |
| 3 | **Focused test: CAT_A vs CAT_B**: match rates, placebo, field-by-field comparison of matched pairs | 8 tables |
| 4 | Scan: which *other* category pairs collide, with placebo lift | 1 table |

PySpark DataFrame API throughout. The only `spark.sql` call is `SHOW PARTITIONS`, which reads the metastore and scans no data.
**Nothing is written.** Everything is shown inline, and ids/accounts are masked unless `SHOW_RAW = True`.

In [ ]:
# ---- Config -------------------------------------------------------------------------------
TABLE     = "neo4j_payments"        # prefix the db if it's not the current database: "<db>.neo4j_payments"
N_MONTHS  = 2                       # latest N complete months found in the partitions
START_DT  = None                    # override the window, e.g. "2026-06-01"
END_DT    = None                    #                       "2026-07-31"  (inclusive)

CAT_A = "5C.RTP_PRT_CPTY_PAYS"
CAT_B = "6C.RTP_P2P_CPTY_PAYS"

# 0 = the duplicate test | ±1 = date-stamp skew between sources | ±7 = placebo (coincidence baseline, same weekday)
DATE_OFFSETS  = [0, -1, 1, -7, 7]
RUN_ALL_PAIRS = True                # §4 scan across every category pair (one extra shuffle join)
MIN_KEYS_PAIR = 100                 # §4: hide category pairs with fewer same-day collisions than this
SHOW_RAW      = False               # True prints unmasked trans_ids / accounts. Never screenshot with True.
SAMPLE_N      = 15

In [ ]:
from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display
from urllib.parse import unquote
import pandas as pd, re, calendar, datetime as dt

spark = SparkSession.builder.getOrCreate()
print("Spark", spark.version, "| pandas", pd.__version__)

pd.set_option("display.max_rows", 200); pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250);    pd.set_option("display.max_colwidth", 80)

def show(pdf, title, pct=(), money=()):
    """Compact, screenshot-friendly table."""
    print(f"\n■ {title}   [{len(pdf):,} rows]")
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.2%}"
        elif c in money:                               fmt[c] = "${:,.0f}"
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.2f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:    sty = sty.hide(axis="index")
    except AttributeError: sty = sty.hide_index()     # pandas < 1.4
    display(sty)

def ints(pdf, cols):
    for c in cols:
        if c in pdf.columns: pdf[c] = pdf[c].fillna(0).astype("int64")
    return pdf

## §0 Schema, partitions, window

In [ ]:
_db, _tbl = TABLE.split(".", 1) if "." in TABLE else (spark.catalog.currentDatabase(), TABLE)
try:    _catalog = spark.catalog.listColumns(f"{_db}.{_tbl}")        # Spark ≥ 3.4 form
except Exception: _catalog = spark.catalog.listColumns(_tbl, _db)     # older Spark
COLTYPE   = {c.name.lower(): c.dataType.lower() for c in _catalog}
PART_COLS = [c.name.lower() for c in _catalog if c.isPartition]

REQUIRED = ["trans_id", "trans_dt", "trans_amt", "category", "src_syst", "payment_rail", "cpty_type",
            "mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_pays", "pnc_dep_acct_receives",
            "customer_name_pays", "customer_name_receives",
            "unq_cpty_acct_id", "cpty_name", "cpty_fin_entity_name"]
missing = [c for c in REQUIRED if c not in COLTYPE]
assert not missing, f"Missing columns in {TABLE}: {missing}"          # fail fast, before any scan

show(pd.DataFrame([{"column": c, "type": COLTYPE[c], "partition": c in PART_COLS}
                   for c in REQUIRED + [p for p in PART_COLS if p not in REQUIRED]]),
     f"{_db}.{_tbl}: columns used")
print("Partition columns:", PART_COLS or "NONE → every query below is a full-table scan")

# Id dtype trap (2026-07 incident class): numeric ids lose leading zeros / precision past 2^53
for c in ["trans_id", "mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_pays", "pnc_dep_acct_receives", "unq_cpty_acct_id"]:
    if not COLTYPE[c].startswith(("string", "varchar", "char")):
        print(f"⚠ {c} is {COLTYPE[c]}: ids should be strings")

In [ ]:
# Partition layout from the metastore. Verifies "partitioned by category + date" without scanning data.
DATE_FMTS = [(r"^\d{4}-\d{2}-\d{2}$", "%Y-%m-%d"), (r"^\d{8}$", "%Y%m%d"),
             (r"^\d{4}-\d{2}$", "%Y-%m"),        (r"^\d{6}$", "%Y%m")]

def parse_part_date(v):
    s = str(v)
    for pat, fmt in DATE_FMTS:
        if re.match(pat, s):
            try:    return dt.datetime.strptime(s, fmt).date(), fmt
            except ValueError: return None, None
    return None, None

PART_PDF, DATE_PART, DATE_PART_FMT = None, None, None
if PART_COLS:
    try:
        specs = [r[0] for r in spark.sql(f"SHOW PARTITIONS {TABLE}").collect()]
        PART_PDF = pd.DataFrame([{k.lower(): unquote(v) for k, v in (kv.split("=", 1) for kv in s.split("/"))}
                                 for s in specs])
    except Exception as e:
        print("SHOW PARTITIONS failed:", str(e)[:300])

if PART_PDF is not None:
    rows = []
    for c in PART_PDF.columns:
        vals    = PART_PDF[c]
        is_date = vals.map(lambda v: parse_part_date(v)[0]).notna().mean() > 0.95
        rows.append({"partition_col": c, "n_distinct": int(vals.nunique()),
                     "min": vals.min(), "max": vals.max(), "looks_like_date": bool(is_date)})
        if is_date and DATE_PART is None:
            DATE_PART, DATE_PART_FMT = c, parse_part_date(vals.dropna().iloc[0])[1]
    print(f"Total partitions: {len(PART_PDF):,}")
    show(pd.DataFrame(rows), "Partition layout (metastore)")
    if DATE_PART:
        d = PART_PDF[DATE_PART].map(lambda v: parse_part_date(v)[0]).dropna()
        pm = pd.Series([x.strftime("%Y-%m") for x in d]).value_counts().sort_index()
        show(pm.tail(15).rename_axis("month").reset_index(name="n_partitions"),
             f"Partitions per month, last 15 (date partition = {DATE_PART})")

In [ ]:
# ---- Window ------------------------------------------------------------------------------
if START_DT is None or END_DT is None:
    assert DATE_PART is not None, "No date partition detected: set START_DT / END_DT explicitly."
    months = sorted({x.replace(day=1) for x in PART_PDF[DATE_PART].map(lambda v: parse_part_date(v)[0]).dropna()})
    months = [m for m in months if m < dt.date.today().replace(day=1)][-N_MONTHS:]   # current month is partial
    assert months, "No complete months in partitions"
    START_DT = months[0].isoformat()
    END_DT   = months[-1].replace(day=calendar.monthrange(months[-1].year, months[-1].month)[1]).isoformat()
print(f"Window: {START_DT} → {END_DT} (inclusive)")

def range_on(col, typ, lo, hi, fmt):
    """Range predicate written in the column's own type/format so it stays a pruning predicate."""
    if typ == "date":
        return F.col(col).between(F.lit(lo).cast("date"), F.lit(hi).cast("date"))
    if typ.startswith("timestamp"):
        return (F.col(col) >= F.lit(lo).cast("timestamp")) & \
               (F.col(col) <  F.date_add(F.lit(hi).cast("date"), 1).cast("timestamp"))
    lo_s = dt.date.fromisoformat(lo).strftime(fmt); hi_s = dt.date.fromisoformat(hi).strftime(fmt)
    if typ in ("int", "bigint", "smallint"):
        return F.col(col).between(int(lo_s), int(hi_s))
    return F.col(col).between(lo_s, hi_s)          # fixed-width date strings sort lexically

cond = range_on("trans_dt", COLTYPE["trans_dt"], START_DT, END_DT, "%Y-%m-%d")
if DATE_PART and DATE_PART != "trans_dt":
    cond = cond & range_on(DATE_PART, COLTYPE[DATE_PART], START_DT, END_DT, DATE_PART_FMT)

raw = spark.table(TABLE).filter(cond)

# Pruning check: an unpruned scan of the whole table is the thing to catch before any action runs
plan = raw._jdf.queryExecution().executedPlan().toString()
pf    = re.findall(r"PartitionFilters: \[([^\]]*)\]", plan)
paths = re.findall(r"FileIndex[^(]*\((\d+) paths\)", plan)
print("\nScan node        :", "HiveTableScan" if "HiveTableScan" in plan else ("FileScan" if "FileScan" in plan else "other"))
print("PartitionFilters :", (pf[0][:400] if pf else "n/a (not a file scan; see plan below)"))
print("Paths read       :", paths[0] if paths else "n/a")
if pf and not pf[0].strip():
    print("⚠ PartitionFilters is EMPTY: the window is not pruning. Check the date partition column / format.")
if not pf:
    print("\n".join(l.strip()[:400] for l in plan.split("\n") if "Scan" in l or "Partition" in l))

In [ ]:
# ---- Base frame: one projection, normalised once, persisted -------------------------------------
def nz(c):
    """String, trimmed, blank → NULL. The table may encode 'absent' either way; keys need one representation."""
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

def shape(c):
    """Masked format signature. Split on '-', each segment → class + length.
    D=digits A=letters X=mixed E=empty.  '071000013-12345678' → 'D9-D8'. Never exposes the value."""
    return F.expr(f"""CASE WHEN {c} IS NULL THEN 'NULL' ELSE array_join(transform(split({c}, '-'), x -> concat(
        CASE WHEN x = '' THEN 'E' WHEN x rlike '^[0-9]+$' THEN 'D' WHEN x rlike '^[A-Za-z]+$' THEN 'A' ELSE 'X' END,
        CAST(length(x) AS STRING))), '-') END""")

base = (raw.select(
            nz("trans_id").alias("trans_id"),
            F.to_date(F.col("trans_dt")).alias("dt"),
            # Key on |amount| in exact cents: never float equality, and robust to one source signing outflows.
            F.abs(F.col("trans_amt").cast("decimal(22,2)")).alias("amt_key"),
            F.abs(F.col("trans_amt").cast("double")).alias("amt"),
            F.signum(F.col("trans_amt").cast("double")).alias("amt_sign"),
            nz("category").alias("category"),       nz("src_syst").alias("src_syst"),
            nz("payment_rail").alias("payment_rail"), nz("cpty_type").alias("cpty_type"),
            nz("mdm_id_pays").alias("mdm_pays"),    nz("mdm_id_receives").alias("mdm_recv"),
            nz("pnc_dep_acct_pays").alias("acct_pays"), nz("pnc_dep_acct_receives").alias("acct_recv"),
            nz("cpty_name").alias("cpty_name"),     nz("unq_cpty_acct_id").alias("cpty_id"),
            nz("cpty_fin_entity_name").alias("cpty_fi"))
        .withColumn("month", F.date_format("dt", "yyyy-MM"))
        # Direction from the null rule, kept independent of category so the two can be cross-checked (§2)
        .withColumn("direction",
                    F.when(F.col("mdm_pays").isNotNull() & F.col("mdm_recv").isNotNull(), "INTERNAL")
                     .when(F.col("mdm_pays").isNull()    & F.col("mdm_recv").isNotNull(), "INBOUND")
                     .when(F.col("mdm_pays").isNotNull() & F.col("mdm_recv").isNull(),    "OUTBOUND")
                     .otherwise("NEITHER"))
        .withColumn("tid_shape", shape("trans_id"))
        .withColumn("cpty_id_shape", shape("cpty_id"))
        .persist(StorageLevel.MEMORY_AND_DISK))
DIRS = ["INTERNAL", "INBOUND", "OUTBOUND", "NEITHER"]

## §1 Window QA

In [ ]:
b = lambda cond: F.sum(cond.cast("long"))
qa = base.agg(
        F.count("*").alias("rows"), F.sum("amt").alias("usd"),
        b(F.col("trans_id").isNull()).alias("null_trans_id"),
        b(F.col("dt").isNull()).alias("unparsed_dt"),
        b(F.col("amt").isNull()).alias("null_amt"),
        b(F.col("amt_sign") < 0).alias("negative_amt"),
        b(F.col("acct_pays").isNull() & F.col("acct_recv").isNull()).alias("no_pnc_acct_either_side"),
        b((F.col("mdm_pays").isNotNull() & F.col("acct_pays").isNull()) |
          (F.col("mdm_recv").isNotNull() & F.col("acct_recv").isNull())).alias("mdm_without_acct"),
        b((F.col("mdm_pays").isNull() & F.col("acct_pays").isNotNull()) |
          (F.col("mdm_recv").isNull() & F.col("acct_recv").isNotNull())).alias("acct_without_mdm"),
     ).toPandas()
WIN_ROWS, WIN_USD = int(qa.rows[0]), float(qa.usd[0])
show(qa, f"Window QA  {START_DT} → {END_DT}", money=("usd",))

# trans_id is asserted unique; verify it on the window
rep = base.filter(F.col("trans_id").isNotNull()).groupBy("trans_id").count().filter("count > 1")
rep_pdf = (base.join(rep.select("trans_id"), "trans_id")
               .groupBy("category").agg(F.count("*").alias("rows_on_repeated_ids"),
                                        F.countDistinct("trans_id").alias("repeated_ids"))
               .orderBy(F.desc("rows_on_repeated_ids")).toPandas())
if len(rep_pdf):
    print("⚠ trans_id is NOT unique in the window")
    show(rep_pdf, "Repeated trans_id by category")
else:
    print("✓ trans_id unique in the window")

## §2 Category inventory

`direction_mixed = True` means a category does not map to a single null pattern. That is worth a look before trusting
category as a substitute for the null rule.

In [ ]:
inv = (base.groupBy("category")
           .agg(F.count("*").alias("rows"), F.sum("amt").alias("usd"),
                *[b(F.col("direction") == d).alias(d) for d in DIRS],
                b(F.col("amt_sign") < 0).alias("neg_amt"),
                F.concat_ws(", ", F.sort_array(F.collect_set("src_syst"))).alias("src_syst"),
                F.concat_ws(", ", F.sort_array(F.collect_set("payment_rail"))).alias("payment_rail"))
           .orderBy("category").toPandas())
inv["row_share"] = inv.rows / inv.rows.sum()
inv["usd_share"] = inv.usd  / inv.usd.sum()
inv["direction_mixed"] = (inv[DIRS] > 0).sum(axis=1) > 1
inv = ints(inv, ["rows", "neg_amt"] + DIRS)
show(inv[["category", "rows", "row_share", "usd", "usd_share"] + DIRS + ["direction_mixed", "neg_amt", "src_syst", "payment_rail"]],
     "Category inventory (direction = null rule on mdm_id_pays / mdm_id_receives)",
     pct=("row_share", "usd_share"), money=("usd",))

bym = base.groupBy("category").pivot("month").count().orderBy("category").toPandas()
show(ints(bym, [c for c in bym.columns if c != "category"]), "Rows per category per month")

## §3 Focused test: CAT_A vs CAT_B

Each row is exploded into its **PNC legs**. An inbound or outbound row has one leg; an internal row has two (payer account and
receiver account). The duplicate key is `(dt, pnc_acct, role, amt_key)`, so a payment one source records as counterparty
and the other as internal still matches on the shared PNC leg.

In [ ]:
leg_arr = F.array(
    F.when(F.col("acct_pays").isNotNull(), F.struct(F.lit("PAYER").alias("role"),    F.col("acct_pays").alias("pnc_acct"))),
    F.when(F.col("acct_recv").isNotNull(), F.struct(F.lit("RECEIVER").alias("role"), F.col("acct_recv").alias("pnc_acct"))))
legs = (base.withColumn("leg", F.explode(leg_arr)).filter(F.col("leg").isNotNull())
            .select("*", "leg.role", "leg.pnc_acct").drop("leg"))
KEY = ["pnc_acct", "role", "amt_key"]          # + dt

present = set(inv.category.dropna())
for c in (CAT_A, CAT_B):
    if c not in present:
        raise ValueError(f"{c} not in window. RTP-like categories present: "
                         f"{sorted(x for x in present if 'RTP' in x.upper())}")

ab = (legs.filter(F.col("category").isin(CAT_A, CAT_B))
          .withColumn("side", F.when(F.col("category") == CAT_A, "A").otherwise("B"))
          .persist(StorageLevel.MEMORY_AND_DISK))

prof = (ab.groupBy("side", "category", "role", "direction")
          .agg(F.count("*").alias("rows"), F.sum("amt").alias("usd"),
               F.countDistinct("pnc_acct").alias("pnc_accts"),
               F.expr("percentile_approx(amt, 0.5)").alias("median_amt"))
          .orderBy("side", "role").toPandas())
show(prof, "A / B profile (which PNC side each category anchors on)", money=("usd",))

In [ ]:
grp = (ab.groupBy("dt", *KEY)
         .agg(F.sum((F.col("side") == "A").cast("int")).alias("n_a"),
              F.sum((F.col("side") == "B").cast("int")).alias("n_b"),
              F.max("amt").alias("amt"))
         .withColumn("k", F.least("n_a", "n_b"))      # rows removable under one-for-one pairing
         .persist(StorageLevel.MEMORY_AND_DISK))

h = grp.agg(
      F.sum("n_a").alias("rows_A"), F.sum("n_b").alias("rows_B"),
      F.sum(F.col("n_a") * F.col("amt")).alias("usd_A"), F.sum(F.col("n_b") * F.col("amt")).alias("usd_B"),
      F.sum(F.when(F.col("n_b") > 0, F.col("n_a")).otherwise(0)).alias("A_matched"),
      F.sum(F.when(F.col("n_b") > 0, F.col("n_a") * F.col("amt")).otherwise(0)).alias("A_matched_usd"),
      F.sum(F.when(F.col("n_a") > 0, F.col("n_b")).otherwise(0)).alias("B_matched"),
      F.sum(F.when(F.col("n_a") > 0, F.col("n_b") * F.col("amt")).otherwise(0)).alias("B_matched_usd"),
      b((F.col("n_a") == 1) & (F.col("n_b") == 1)).alias("pairs_1to1"),
      b((F.col("k") >= 1) & ((F.col("n_a") > 1) | (F.col("n_b") > 1))).alias("groups_m_n"),
      b(F.col("n_a") >= 2).alias("A_repeat_groups"), b(F.col("n_b") >= 2).alias("B_repeat_groups"),
      F.sum("k").alias("dup_rows"), F.sum(F.col("k") * F.col("amt")).alias("dup_usd"),
    ).first().asDict()

hl = pd.DataFrame([
  {"metric": f"A rows  ({CAT_A})",                    "count": h["rows_A"], "pct": None, "usd": h["usd_A"], "pct_usd": None},
  {"metric": f"B rows  ({CAT_B})",                    "count": h["rows_B"], "pct": None, "usd": h["usd_B"], "pct_usd": None},
  {"metric": "A rows with a same-day B match",          "count": h["A_matched"], "pct": h["A_matched"] / max(h["rows_A"], 1),
   "usd": h["A_matched_usd"], "pct_usd": h["A_matched_usd"] / max(h["usd_A"], 1)},
  {"metric": "B rows with a same-day A match",          "count": h["B_matched"], "pct": h["B_matched"] / max(h["rows_B"], 1),
   "usd": h["B_matched_usd"], "pct_usd": h["B_matched_usd"] / max(h["usd_B"], 1)},
  {"metric": "clean 1:1 groups (one A, one B)",         "count": h["pairs_1to1"], "pct": None, "usd": None, "pct_usd": None},
  {"metric": "m:n groups (both sides, repeats inside)", "count": h["groups_m_n"], "pct": None, "usd": None, "pct_usd": None},
  {"metric": "A-only repeat groups (≥2 A same key)",    "count": h["A_repeat_groups"], "pct": None, "usd": None, "pct_usd": None},
  {"metric": "B-only repeat groups (≥2 B same key)",    "count": h["B_repeat_groups"], "pct": None, "usd": None, "pct_usd": None},
  {"metric": "EST. DUPLICATE ROWS  Σ min(n_A, n_B)  (pct = of ALL window rows / $)",
   "count": h["dup_rows"], "pct": h["dup_rows"] / WIN_ROWS, "usd": h["dup_usd"], "pct_usd": h["dup_usd"] / WIN_USD},
])
show(ints(hl, ["count"]), "Headline: same-day A↔B matches", pct=("pct", "pct_usd"), money=("usd",))

ct = (grp.withColumn("A_in_group", F.when(F.col("n_a") >= 3, "3+").otherwise(F.col("n_a").cast("string")))
         .withColumn("B_in_group", F.when(F.col("n_b") >= 3, "3+").otherwise(F.col("n_b").cast("string")))
         .groupBy("A_in_group").pivot("B_in_group", ["0", "1", "2", "3+"]).count()
         .orderBy("A_in_group").toPandas())
show(ints(ct, ["0", "1", "2", "3+"]), "Key-group shape: rows = A per group, cols = B per group (cell = # groups)")

In [ ]:
# ---- Placebo / date-skew test -------------------------------------------------------------
# Same-day match rate alone cannot separate duplicates from genuine same-amount repeats.
# Shifting B by ±7 days (same weekday) estimates the coincidence rate; ±1 detects sources stamping different dates.
a_rows = (ab.filter(F.col("side") == "A").select("dt", "amt", *KEY)
            .withColumn("rid", F.monotonically_increasing_id()).persist(StorageLevel.MEMORY_AND_DISK))
b_keys = ab.filter(F.col("side") == "B").select("dt", *KEY).distinct().persist(StorageLevel.MEMORY_AND_DISK)
nA = a_rows.count()

def matched(k):   # A rows on day d whose key exists in B on day d-k
    return a_rows.join(b_keys.withColumn("dt", F.date_add("dt", k)), ["dt"] + KEY, "left_semi")

m0_ids = matched(0).select("rid").persist(StorageLevel.MEMORY_AND_DISK)
label = {0: "same day: THE DUPLICATE TEST", 1: "date skew", -1: "date skew", 7: "placebo", -7: "placebo"}
rows = []
for k in DATE_OFFSETS:
    mk = matched(k)
    r  = mk.agg(F.count("*").alias("n"), F.sum("amt").alias("usd")).first()
    new = r["n"] if k == 0 else mk.join(m0_ids, "rid", "left_anti").count()
    rows.append({"B_shift_days": k, "reads_as": label.get(k, ""), "A_rows_matched": r["n"],
                 "pct_of_A": r["n"] / max(nA, 1), "usd": r["usd"] or 0.0,
                 "matched_ONLY_at_this_shift": new, "pct_only": new / max(nA, 1)})
off = pd.DataFrame(rows)
show(off, f"A rows matching B at a date shift  (A rows = {nA:,})", pct=("pct_of_A", "pct_only"), money=("usd",))

p0 = off.loc[off.B_shift_days == 0, "pct_of_A"].iloc[0]
pp = off.loc[off.B_shift_days.abs() == 7, "pct_of_A"].mean()
print(f"Same-day {p0:.2%}  vs placebo ±7d {pp:.2%}  →  excess (duplicate signal) ≈ {p0 - pp:.2%} of A rows, "
      f"lift {p0 / pp:,.1f}×" if pp > 0 else f"Same-day {p0:.2%}, placebo 0 → every same-day match is excess")

### Matched 1:1 pairs, field by field

This is restricted to clean 1:1 groups so every A row has exactly one partner. These tables answer three things: which source carries
the counterparty name, id, and FI; whether the two ids are related; and whether `src_syst` separates the sources.

In [ ]:
CMP = ["trans_id", "tid_shape", "src_syst", "payment_rail", "cpty_type", "direction", "amt_sign",
       "mdm_pays", "mdm_recv", "acct_pays", "acct_recv", "cpty_name", "cpty_id", "cpty_id_shape", "cpty_fi"]
CASE_INSENSITIVE = {"cpty_name", "cpty_fi"}

g11 = grp.filter((F.col("n_a") == 1) & (F.col("n_b") == 1)).select("dt", *KEY)
def side_cols(s):
    return (ab.filter(F.col("side") == s).join(g11, ["dt"] + KEY)
              .select("dt", *KEY, *[F.col(c).alias(f"{c}_{s.lower()}") for c in CMP]))
pairs = side_cols("A").join(side_cols("B"), ["dt"] + KEY).persist(StorageLevel.MEMORY_AND_DISK)
n_pairs = pairs.count()

def status(c):
    a, bb = F.col(f"{c}_a").cast("string"), F.col(f"{c}_b").cast("string")
    if c in CASE_INSENSITIVE: a, bb = F.upper(a), F.upper(bb)
    return (F.when(a.isNull() & bb.isNull(), "both_null").when(a.isNull(), "A_null")
             .when(bb.isNull(), "B_null").when(a == bb, "equal").otherwise("differ"))

ST = ["equal", "differ", "A_null", "B_null", "both_null"]
agree = (pairs.select(F.explode(F.array(*[F.struct(F.lit(c).alias("field"), status(c).alias("status")) for c in CMP])).alias("s"))
              .select("s.field", "s.status").groupBy("field").pivot("status", ST).count().toPandas()
              .set_index("field").reindex(CMP).reset_index())
agree = ints(agree, ST)
for c in ST: agree[c] = agree[c] / max(n_pairs, 1)
show(agree, f"Field agreement across 1:1 pairs (n = {n_pairs:,}); share of pairs", pct=tuple(ST))

In [ ]:
rel = (pairs.withColumn("tid_relation", F.expr("""
          CASE WHEN trans_id_a IS NULL OR trans_id_b IS NULL                    THEN 'one side null'
               WHEN trans_id_a = trans_id_b                                     THEN 'identical'
               WHEN instr(trans_id_a, trans_id_b) > 0 OR instr(trans_id_b, trans_id_a) > 0
                                                                                THEN 'one contains the other'
               WHEN split(trans_id_a, '-')[0] = split(trans_id_b, '-')[0]       THEN 'same first segment'
               WHEN right(trans_id_a, 8) = right(trans_id_b, 8)                 THEN 'same last 8 chars'
               ELSE 'unrelated' END"""))
            .groupBy("tid_relation").count().orderBy(F.desc("count")).toPandas())
rel["pct"] = rel["count"] / max(n_pairs, 1)
show(rel, "trans_id relationship within 1:1 pairs", pct=("pct",))

sh = (pairs.groupBy("tid_shape_a", "tid_shape_b").count().orderBy(F.desc("count")).limit(20).toPandas())
sh["pct"] = sh["count"] / max(n_pairs, 1)
show(sh, "trans_id format: A × B (top 20)", pct=("pct",))

ss = (pairs.groupBy("src_syst_a", "src_syst_b", "payment_rail_a", "payment_rail_b").count()
           .orderBy(F.desc("count")).limit(20).toPandas())
ss["pct"] = ss["count"] / max(n_pairs, 1)
show(ss, "src_syst / payment_rail: A × B (top 20)", pct=("pct",))

# id formats of the full A and B populations, to compare matched vs unmatched
fmt_all = (ab.groupBy("side", "tid_shape").count()
             .withColumn("pct_of_side", F.col("count") / F.sum("count").over(Window.partitionBy("side")))
             .orderBy("side", F.desc("count")).toPandas())
show(fmt_all.groupby("side").head(8), "trans_id formats per side, all A / B rows (top 8 each)", pct=("pct_of_side",))

In [ ]:
def mask(c):
    return F.when(F.col(c).isNull(), F.lit(None)).otherwise(F.concat(F.lit("…"), F.substring(F.col(c), -4, 4)))

cols = [F.col("dt").cast("string").alias("dt"), F.col("amt_key").cast("double").alias("amount"), "role",
        (F.col("pnc_acct") if SHOW_RAW else mask("pnc_acct")).alias("pnc_acct"),
        "tid_shape_a", "tid_shape_b", "src_syst_a", "src_syst_b",
        status("cpty_name").alias("cpty_name"), status("cpty_id").alias("cpty_id"), status("cpty_fi").alias("cpty_fi")]
if SHOW_RAW: cols += ["trans_id_a", "trans_id_b"]
show(pairs.select(*cols).orderBy(F.rand(42)).limit(SAMPLE_N).toPandas(),
     f"Random 1:1 pairs ({'RAW' if SHOW_RAW else 'masked'})", money=("amount",))

## §4 Which other category pairs collide?

This runs the same key over every category. `lift` = same-day match rate ÷ the rate with one side shifted by 7 days. A lift near 1 is
coincidence (recurring same-amount payments). A large lift is a duplicate candidate: point CAT_A/CAT_B at it and re-run §3.

In [ ]:
if RUN_ALL_PAIRS:
    kc = (legs.groupBy("dt", *KEY, "category").agg(F.count("*").alias("n"))
              .persist(StorageLevel.MEMORY_AND_DISK))       # aggregate first: no row-level blow-up on busy accounts
    nk = kc.groupBy("category").count().toPandas().set_index("category")["count"]

    def collide(k):
        x = kc.select("dt", *KEY, F.col("category").alias("cat_1"), F.col("n").alias("n_1"))
        y = kc.select(F.date_add("dt", k).alias("dt"), *KEY, F.col("category").alias("cat_2"), F.col("n").alias("n_2"))
        return (x.join(y, ["dt"] + KEY).filter(F.col("cat_1") < F.col("cat_2"))
                 .groupBy("cat_1", "cat_2")
                 .agg(F.count("*").alias("keys"),
                      b((F.col("n_1") == 1) & (F.col("n_2") == 1)).alias("keys_1to1"),
                      F.sum(F.least("n_1", "n_2")).alias("dup_rows"),
                      F.sum(F.least("n_1", "n_2") * F.col("amt_key").cast("double")).alias("dup_usd")))

    c0 = collide(0).toPandas()
    c7 = collide(7).select("cat_1", "cat_2", F.col("keys").alias("keys_placebo")).toPandas()
    allp = c0.merge(c7, on=["cat_1", "cat_2"], how="left").fillna({"keys_placebo": 0})
    allp["denom"]        = [min(nk.get(a, 0), nk.get(c, 0)) for a, c in zip(allp.cat_1, allp.cat_2)]
    allp["match_rate"]   = allp["keys"] / allp["denom"].clip(lower=1)
    allp["placebo_rate"] = allp["keys_placebo"] / allp["denom"].clip(lower=1)
    allp["lift"]         = allp["match_rate"] / allp["placebo_rate"].where(allp["placebo_rate"] > 0)
    allp["pct_window_usd"] = allp["dup_usd"] / WIN_USD
    allp = ints(allp, ["keys", "keys_1to1", "dup_rows", "keys_placebo", "denom"])
    show(allp[allp["keys"] >= MIN_KEYS_PAIR].sort_values("dup_rows", ascending=False).head(30)
             [["cat_1", "cat_2", "keys", "keys_1to1", "match_rate", "placebo_rate", "lift", "dup_rows", "dup_usd", "pct_window_usd"]],
         f"Category pairs colliding on (dt, pnc_acct, role, amount): top 30 by est. duplicate rows (keys ≥ {MIN_KEYS_PAIR})",
         pct=("match_rate", "placebo_rate", "pct_window_usd"), money=("dup_usd",))
    print("lift = — : placebo found zero matches (every same-day match is excess)")

In [ ]:
# ---- Release cached frames ----------------------------------------------------------------
for _df in ["kc", "pairs", "m0_ids", "b_keys", "a_rows", "grp", "ab", "base"]:
    if _df in globals(): globals()[_df].unpersist()

### What to screenshot
§0 partition layout · §1 window QA · §2 category inventory · §3 headline, key-group shape, **date-shift/placebo table**,
field agreement, trans_id relationship, formats A × B, src_syst A × B · §4 category-pair table.